# Code Generation: Training GPT-2 dari Nol

Notebook ini melatih model GPT-2 **dari nol** (bukan fine-tuning) untuk menyelesaikan potongan kode Python — studi kasus "Code Generation" dari materi NLP Text Generation.

**Alur:** filter dataset kode → tokenisasi → bangun arsitektur GPT-2 kosong → training dengan `Trainer` → uji coba melengkapi kode.

> Bagian deployment (menjadikan model ini sebuah API endpoint) ada di notebook terpisah: **`02_deploy_text_generation_api.ipynb`**.

In [7]:
!pip install transformers[torch] datasets evaluate accelerate

## 1. Filtering Dataset

Dataset sumber (`codeparrot`) berisi kode Python dari banyak library. Kita filter agar hanya baris kode yang relevan dengan data science (numpy, pandas, sklearn, dst) yang dipakai.

In [8]:
def any_keyword_in_string(string, keywords):
    for keyword in keywords:
        if keyword in string:
            return True
    return False

In [9]:
filters = ["numpy", "pandas", "sklearn", "matplotlib", "seaborn"]
example_1 = "import numpy as np"
example_2 = "import pandas as pd"
example_3 = "import pytorch as torch"

print(
    any_keyword_in_string(example_1, filters),
    any_keyword_in_string(example_2, filters),
    any_keyword_in_string(example_3, filters),
)

True True False


In [10]:
from collections import defaultdict
from tqdm import tqdm
from datasets import Dataset


def filter_streaming_dataset(dataset, filters):
    filtered_dict = defaultdict(list)
    total = 0
    for sample in tqdm(iter(dataset)):
        total += 1
        if any_keyword_in_string(sample["content"], filters):
            for k, v in sample.items():
                filtered_dict[k].append(v)
    print(f"{len(filtered_dict['content'])/total:.2%} of data after filtering.")
    return Dataset.from_dict(filtered_dict)

> **Opsional — lompati cell ini.** Cell di bawah menjalankan filtering dari nol pada dataset `codeparrot` mentah (~50GB, ratusan file, bisa memakan waktu sangat lama). Untuk latihan, langsung pakai dataset yang sudah difilter di cell berikutnya (`huggingface-course/codeparrot-ds-*`).\n\n> **Penting:** `load_dataset(..., streaming=True)` wajib dipakai di sini — tanpa itu, Colab akan mencoba mengunduh & menulis seluruh ~50GB dataset mentah ke disk sebelum sempat difilter, dan biasanya berakhir dengan error `OSError: No space left on device`. Dengan `streaming=True`, data diproses baris-per-baris (mode `IterableDataset`) tanpa pernah tersimpan penuh ke disk.

In [ ]:
# OPSIONAL — proses filtering dari dataset mentah, sangat lama. Skip ke cell berikutnya untuk latihan.
from datasets import load_dataset

split = "train"  # "valid"
filters = ["pandas", "sklearn", "matplotlib", "seaborn"]

data = load_dataset(f"transformersbook/codeparrot-{split}", split=split, streaming=True)
filtered_data = filter_streaming_dataset(data, filters)

In [11]:
from datasets import load_dataset, DatasetDict

ds_train = load_dataset("huggingface-course/codeparrot-ds-train", split="train")
ds_valid = load_dataset("huggingface-course/codeparrot-ds-valid", split="validation")

raw_datasets = DatasetDict(
    {
        "train": ds_train.shuffle().select(range(50000)),
        "valid": ds_valid.shuffle().select(range(500)),
    }
)

raw_datasets

codeparrot-ds-train.jsonl: reconstructing file:   0%|          |  0.00B / 8.25GB            

codeparrot-ds-train.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

codeparrot-ds-valid.jsonl: reconstructing file:   0%|          |  0.00B / 46.1MB            

codeparrot-ds-valid.jsonl: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/3322 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'path', 'copies', 'size', 'content', 'license'],
        num_rows: 50000
    })
    valid: Dataset({
        features: ['repo_name', 'path', 'copies', 'size', 'content', 'license'],
        num_rows: 500
    })
})

In [12]:
for key in raw_datasets["train"][0]:
    print(f"{key.upper()}: {raw_datasets['train'][0][key][:200]}")

REPO_NAME: abigailStev/stingray
PATH: stingray/bispectrum.py
COPIES: 2
SIZE: 16265
CONTENT: from __future__ import division

import numpy as np
from scipy.linalg import toeplitz
from scipy.fftpack import fftshift, fft2, ifftshift, fft
from  scipy.linalg import hankel

from stingray import li
LICENSE: mit


## 2. Tokenisasi

Setiap dokumen kode dipotong (`truncation`) menjadi chunk sepanjang `context_length` token.

In [13]:
from transformers import AutoTokenizer

context_length = 128
tokenizer = AutoTokenizer.from_pretrained("huggingface-course/code-search-net-tokenizer")

outputs = tokenizer(
    raw_datasets["train"][:2]["content"],
    truncation=True,
    max_length=context_length,
    return_overflowing_tokens=True,
    return_length=True,
)

print(f"Input IDs length: {len(outputs['input_ids'])}")
print(f"Input chunk lengths: {(outputs['length'])}")
print(f"Chunk mapping: {outputs['overflow_to_sample_mapping']}")

tokenizer_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/789k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/448k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

Input IDs length: 2
Input chunk lengths: [128, 128]
Chunk mapping: [0, 1]


In [14]:
def tokenize(element):
    outputs = tokenizer(
        element["content"],
        truncation=True,
        max_length=context_length,
        return_overflowing_tokens=True,
        return_length=True,
    )
    input_batch = []
    for length, input_ids in zip(outputs["length"], outputs["input_ids"]):
        if length == context_length:
            input_batch.append(input_ids)
    return {"input_ids": input_batch}


tokenized_datasets = raw_datasets.map(
    tokenize, batched=True, remove_columns=raw_datasets["train"].column_names
)
tokenized_datasets

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 50000
    })
    valid: Dataset({
        features: ['input_ids'],
        num_rows: 500
    })
})

## 3. Membangun Arsitektur GPT-2 dari Nol

Bedanya dengan fine-tuning: kita pakai `AutoConfig` untuk mendefinisikan arsitektur GPT-2, lalu membuat model **dengan bobot acak** (`GPT2LMHeadModel(config)`), bukan memuat bobot pretrained.

In [15]:
from transformers import AutoTokenizer, GPT2LMHeadModel, AutoConfig

config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),
    n_ctx=context_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [16]:
model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"GPT-2 size: {model_size/1000**2:.1f}M parameters")

GPT-2 size: 124.2M parameters


## 4. Training

In [17]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [21]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="codeparrot-ds",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="steps",
    eval_steps=5_000,
    logging_steps=5_000,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    weight_decay=0.1,
    warmup_steps=1_000,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    save_steps=5_000,
    fp16=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
)

In [22]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
196,No log,3.975246


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=196, training_loss=6.434496821189414, metrics={'train_runtime': 632.0501, 'train_samples_per_second': 79.108, 'train_steps_per_second': 0.31, 'total_flos': 3266150400000000.0, 'train_loss': 6.434496821189414, 'epoch': 1.0})

### Simpan Model ke Google Drive

**Penting:** penyimpanan `output_dir="codeparrot-ds"` di atas hanya tersimpan di storage sementara Colab (hilang begitu runtime berakhir/direset). Supaya model ini bisa dipakai lagi nanti — termasuk untuk di-deploy sebagai API di `02_deploy_text_generation_api.ipynb` — simpan eksplisit ke Google Drive di sini.

In [24]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = "/content/drive/MyDrive/text-generation-project/code-generator-model"
os.makedirs(save_dir, exist_ok=True)

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model & tokenizer tersimpan di: {save_dir}")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model & tokenizer tersimpan di: /content/drive/MyDrive/text-generation-project/code-generator-model


## 5. Uji Coba Code Completion

In [25]:
import torch
from transformers import pipeline

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer, device=device
)

In [26]:
txt = """\
# create some data
x = np.random.randn(100)
y = np.random.randn(100)

# create scatter plot with x, y
"""
print(pipe(txt, num_return_sequences=1)[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


# create some data
x = np.random.randn(100)
y = np.random.randn(100)

# create scatter plot with x, y
#  #
# This file is a f:
# This is a #
#
# Author:
# This program is free software: you can redistribute it and/or modify
# the terms of the GNU General Public License as published by
# it and/or modify
# (at your option) any later version 3 of the License as published by
# (data) any later version.
#  WITHOUT ANY WITHOUT ANY WITHOUT WARRANTY;
#-2, either version 3,
#
#   http://www.
#
#
# it will be useful,
    from time full (at your option) any later version 3.
# the terms of the License,
# with
# This program is distributed in the License is distributed in the License is distributed in the License,
# (the "AS IS"
# the License, or
#
# Copyright (at your option) any later version.  See the License is distributed in the License, either version 3 of the License,
# will be useful,
#
#
#
# (the "AS IS" BASIS, or agreed to the License, either version 3 of the


In [27]:
txt = """\
# create some data
x = np.random.randn(100)
y = np.random.randn(100)

# create dataframe from x and y
"""
print(pipe(txt, num_return_sequences=1)[0]["generated_text"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


# create some data
x = np.random.randn(100)
y = np.random.randn(100)

# create dataframe from x and y
# the of the data
# with the terms of the
# (C)
# This is the terms of the terms of the Free Software Foundation, and the terms of the Free Software Foundation, or (at your option) (at your option) any later version.
# the hope that it will be useful, or
# but WITHOUT ANY WARRANTIES OR CONDITIONS OF ANY WITHOUT ANY WARRANTY; without even the hope that it and/or modify
# (at your option) any later version 2, or (at your option) any later version 3, or
# the GNU General Public License,
# NEST is distributed in the License, or
# the hope that it will be useful,
#
# WARRANTY; without even the hope that it will be useful, either version 3 of the License, or
#
# WARRANTY; without even the License,
# (at your option) any later version 2 of the hope that it will be useful,
# the License,
# it will be useful, or (at your option) any later version 3 as published by the License as published by
# 

---
*Materi: rubythalib.ai AI Engineer Bootcamp — Modul NLP (mentor: Muhammad Ikhwan Fathulloh). Notebook ini adalah versi konsolidasi/rapi dari catatan latihan sesi Text Generation, disusun ulang untuk dokumentasi pribadi.*